In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics.pairwise import cosine_similarity

from config import CHROMA_DIR, TOP_K_RETRIEVAL
from rag.embedder  import get_embedding_function
from rag.store     import get_chroma_collection
from rag.retriever import retrieve_context

embed_fn   = get_embedding_function()
collection = get_chroma_collection(CHROMA_DIR)
print(f'Docs in collection: {collection.count()}')

In [ ]:
# ← Modify these queries to compare semantic similarity
queries = [
    'What programs does Pradita University offer?',
    'What are the study programs at Pradita?',
    'How do I register as a new student?',
    'What is the tuition fee?',
    'Tell me about the campus location',
]

# Embed all queries
vectors = np.array([embed_fn.embed_query(q) for q in queries])

# Compute cosine similarity matrix
sim_matrix = cosine_similarity(vectors)

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)

short_labels = [q[:35] + '...' if len(q) > 35 else q for q in queries]
ax.set_xticks(range(len(queries)))
ax.set_yticks(range(len(queries)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

for i in range(len(queries)):
    for j in range(len(queries)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}',
                ha='center', va='center',
                color='black' if sim_matrix[i,j] < 0.7 else 'white', fontsize=9)

plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Query–Query Cosine Similarity Matrix', pad=15, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('similarity_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → similarity_matrix.png')

In [ ]:
# Test retrieval for each query and display scores
test_query = queries[0]  # ← change index to test different queries

result = retrieve_context(test_query, top_k=5, collection=collection)

print(f'Query: "{test_query}"\n')
print(f'{"Rank":<5} {"Distance":<10} {"Source":<30} {"Preview"}')
print('-' * 80)

for rank, chunk in enumerate(result['chunks'], 1):
    preview = chunk['text'][:60].replace('\n', ' ') + '...'
    print(f"{rank:<5} {chunk['distance']:<10.4f} {chunk['source']:<30} {preview}")

# Distance distribution bar chart
distances  = [c['distance'] for c in result['chunks']]
rank_labels = [f'Rank {i+1}' for i in range(len(distances))]

fig, ax = plt.subplots(figsize=(7, 4))
colors = cm.RdYlGn_r(np.linspace(0.1, 0.9, len(distances)))
bars = ax.bar(rank_labels, distances, color=colors, edgecolor='white', linewidth=0.5)

for bar, d in zip(bars, distances):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{d:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Cosine Distance (lower = more similar)')
ax.set_title(f'Retrieval Distances for: "{test_query[:40]}..."', fontsize=11)
ax.set_ylim(0, min(1.1, max(distances) * 1.3))
plt.tight_layout()
plt.show()

In [ ]:
# Fetch all stored chunks from ChromaDB
if collection.count() == 0:
    print('⚠️ Collection is empty. Run the ingestion first.')
else:
    all_data = collection.get(include=['embeddings', 'documents', 'metadatas'])
    all_vecs = np.array(all_data['embeddings'])
    all_srcs = [m.get('source', 'unknown') for m in all_data['metadatas']]
    all_texts = all_data['documents']

    print(f'Total vectors fetched : {len(all_vecs)}')
    print(f'Vector dimensions     : {all_vecs.shape[1]}')
    print(f'Unique sources        : {set(all_srcs)}')

In [ ]:
# Run t-SNE on stored chunk embeddings
# (Install umap-learn for UMAP: pip install umap-learn)
from sklearn.manifold import TSNE

if collection.count() > 1:
    n_components = min(2, len(all_vecs) - 1)
    perplexity   = min(30, len(all_vecs) - 1)

    tsne = TSNE(n_components=2, perplexity=perplexity,
                random_state=42, max_iter=1000, verbose=0)
    reduced = tsne.fit_transform(all_vecs)

    # Colour by source file
    unique_srcs = list(set(all_srcs))
    colors_map  = {s: cm.tab10(i / len(unique_srcs)) for i, s in enumerate(unique_srcs)}
    point_colors = [colors_map[s] for s in all_srcs]

    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(reduced[:, 0], reduced[:, 1],
                         c=point_colors, alpha=0.7, s=20, edgecolors='none')

    # Legend patches
    import matplotlib.patches as mpatches
    patches = [mpatches.Patch(color=colors_map[s], label=s) for s in unique_srcs]
    ax.legend(handles=patches, title='PDF Source', loc='best', fontsize=8)

    # Overlay query positions
    for q in queries[:3]:
        q_vec = np.array([embed_fn.embed_query(q)])
        # We can't directly run t-SNE on a single point;
        # approximate by finding nearest stored vector
        sims  = cosine_similarity(q_vec, all_vecs)[0]
        nearest_idx = sims.argmax()
        ax.scatter(reduced[nearest_idx, 0], reduced[nearest_idx, 1],
                   marker='*', s=200, zorder=5, color='red')
        ax.annotate(q[:25], xy=(reduced[nearest_idx, 0], reduced[nearest_idx, 1]),
                    fontsize=7, color='red')

    ax.set_title('t-SNE Embedding Visualization of RAG Chunks', fontsize=13, fontweight='bold')
    ax.set_xlabel('t-SNE Dim 1')
    ax.set_ylabel('t-SNE Dim 2')
    plt.tight_layout()
    plt.savefig('tsne_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → tsne_embeddings.png')
else:
    print('⚠️ Need at least 2 chunks for t-SNE.')